In [23]:
risk_raw = [c for c in df.columns if c.startswith("risk_") and "(t-1)" not in c]
df = df.sort_values(["matched_admin1_id","month_year"])

for c in risk_raw:
    df[f"{c} (t-1)"] = df.groupby("matched_admin1_id")[c].shift(1).fillna(0).astype(int)

# check totals
risk_lag = [f"{c} (t-1)" for c in risk_raw]
print(df[risk_lag].sum().sort_values(ascending=False))

df.to_csv("../../data/processed/model_data_full.csv", index=False)
print("saved")


risk_crop_failure (t-1)               105910
risk_natural_disaster (t-1)            94828
risk_economic_concern (t-1)            68805
risk_ethnic_tension (t-1)              38054
risk_political_assassination (t-1)     27770
risk_contested_election (t-1)          27736
risk_military_coup (t-1)               18908
dtype: int64
saved


In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../../data/processed/model_data_full.csv")

print("="*60)
print("BASIC INFO")
print("="*60)
print("Shape:", df.shape)
print("Date range:", df["month_year"].min(), "→", df["month_year"].max())
print("Unique admin1:", df["matched_admin1_id"].nunique())
print("Unique countries:", df["matched_admin1_id"].str.split(" - ").str[0].nunique())

print("\nMissing values (%)")
print((df.isna().mean()*100).sort_values(ascending=False).head(20))

BASIC INFO
Shape: (278820, 60)
Date range: 2018-01-01 → 2025-06-01
Unique admin1: 3097
Unique countries: 220

Missing values (%)
income_inequality                              66.115774
youth_unemployment                             55.713363
inflation                                      32.111039
income_level_code                              30.837099
Excessive force against protesters (t-1)        1.143031
Agreement (t-1)                                 1.143031
Strategic developments (t-1)                    1.143031
Violence against civilians (t-1)                1.143031
Explosions/Remote violence (t-1)                1.143031
Battles (t-1)                                   1.143031
Riots (t-1)                                     1.143031
Protests (t-1)                                  1.143031
Battles_neighbours (t-1)                        1.143031
Riots_neighbours (t-1)                          1.143031
Explosions/Remote violence_lag                  1.143031
Battles_lag     

In [25]:
#Target Distribution:


print("="*60)
print("TARGET DISTRIBUTION")
print("="*60)

targets = [
    "Battles",
    "Explosions/Remote violence",
    "Violence against civilians"
]

for t in targets:
    if t in df.columns:
        print(f"\n--- {t} ---")
        print("Mean:", df[t].mean())
        print("Median:", df[t].median())
        print("% Zeros:", (df[t] == 0).mean() * 100)

TARGET DISTRIBUTION

--- Battles ---
Mean: 1.236094971666308
Median: 0.0
% Zeros: 88.40434689046697

--- Explosions/Remote violence ---
Mean: 1.412172727924826
Median: 0.0
% Zeros: 94.29596155225593

--- Violence against civilians ---
Mean: 0.8532709274800947
Median: 0.0
% Zeros: 83.87023886378309


In [26]:
print("="*60)
print("TARGET PERSISTENCE CHECK")
print("="*60)

for t in ["Battles", "Explosions/Remote violence", "Violence against civilians"]:
    if t in df.columns:
        df[f"{t}_lag"] = df.groupby("matched_admin1_id")[t].shift(1)
        persistence = ((df[t] > 0) & (df[f"{t}_lag"] > 0)).mean()
        print(f"{t} persistence rate:", persistence)

TARGET PERSISTENCE CHECK
Battles persistence rate: 0.083480381608206
Explosions/Remote violence persistence rate: 0.03949501470482749
Violence against civilians persistence rate: 0.10650598952729359


In [27]:
print("="*60)
print("EVENT ACTIVITY PER ADMIN1")
print("="*60)

activity = df.groupby("matched_admin1_id")[["Battles","Explosions/Remote violence","Violence against civilians"]].sum()

print("Mean total events per admin1:")
print(activity.mean())

print("\nMedian total events per admin1:")
print(activity.median())

print("\n% admin1 with zero total Battles:")
print((activity["Battles"] == 0).mean() * 100)

print("\n% admin1 with zero total Violence against civilians:")
print((activity["Violence against civilians"] == 0).mean() * 100)

EVENT ACTIVITY PER ADMIN1
Mean total events per admin1:
Battles                       111.284469
Explosions/Remote violence    127.136584
Violence against civilians     76.819180
dtype: float64

Median total events per admin1:
Battles                       0.0
Explosions/Remote violence    0.0
Violence against civilians    2.0
dtype: float64

% admin1 with zero total Battles:
56.893768162738134

% admin1 with zero total Violence against civilians:
32.25702292541169


In [28]:
regions_to_check = [
    "USA - Alabama",
    "USA - Florida",
    "UKR - Cherkasy",
    "UKR - Zakarpattia",
]

for r in regions_to_check:
    subset = df[df["matched_admin1_id"] == r]
    print("\n", r)
    print("Rows:", len(subset))
    print("Total Battles:", subset["Battles"].sum())
    print("Total Explosions/Remote violence:", subset["Explosions/Remote violence"].sum())
    print("Total Violence against civilians:", subset["Violence against civilians"].sum())
    print("Avg Protests (t-1):", subset["Protests (t-1)"].mean())
    print("Avg Riots (t-1):", subset["Riots (t-1)"].mean())



 USA - Alabama
Rows: 90
Total Battles: 0
Total Explosions/Remote violence: 1
Total Violence against civilians: 8
Avg Protests (t-1): 8.797752808988765
Avg Riots (t-1): 0.10112359550561797

 USA - Florida
Rows: 90
Total Battles: 0
Total Explosions/Remote violence: 0
Total Violence against civilians: 32
Avg Protests (t-1): 42.97752808988764
Avg Riots (t-1): 0.5393258426966292

 UKR - Cherkasy
Rows: 90
Total Battles: 0
Total Explosions/Remote violence: 109
Total Violence against civilians: 5
Avg Protests (t-1): 1.8314606741573034
Avg Riots (t-1): 0.14606741573033707

 UKR - Zakarpattia
Rows: 90
Total Battles: 0
Total Explosions/Remote violence: 10
Total Violence against civilians: 6
Avg Protests (t-1): 0.9438202247191011
Avg Riots (t-1): 0.12359550561797752


In [29]:
print("="*60)
print("RISK FEATURE ACTIVITY")
print("="*60)

risk_cols = [c for c in df.columns if c.startswith("risk_") and "(t-1)" in c]

risk_activity = df[risk_cols].sum()

print("Total mentions per risk feature:")
print(risk_activity.sort_values(ascending=False).head(10))

print("\nFeatures with almost zero total activity:")
print(risk_activity.sort_values().head(10))

RISK FEATURE ACTIVITY
Total mentions per risk feature:
risk_crop_failure (t-1)               105910
risk_natural_disaster (t-1)            94828
risk_economic_concern (t-1)            68805
risk_ethnic_tension (t-1)              38054
risk_political_assassination (t-1)     27770
risk_contested_election (t-1)          27736
risk_military_coup (t-1)               18908
dtype: int64

Features with almost zero total activity:
risk_military_coup (t-1)               18908
risk_contested_election (t-1)          27736
risk_political_assassination (t-1)     27770
risk_ethnic_tension (t-1)              38054
risk_economic_concern (t-1)            68805
risk_natural_disaster (t-1)            94828
risk_crop_failure (t-1)               105910
dtype: int64


In [30]:
risk_cols_raw = [c for c in df.columns if c.startswith("risk_") and "(t-1)" not in c]

print("Raw risk feature totals:")
print(df[risk_cols_raw].sum())

Raw risk feature totals:
risk_contested_election          28575
risk_crop_failure               107808
risk_economic_concern            69813
risk_ethnic_tension              38162
risk_military_coup               19183
risk_natural_disaster            96994
risk_political_assassination     29309
dtype: int64


In [31]:
print("="*60)
print("TIME CONTINUITY CHECK")
print("="*60)

expected_months = df["month_year"].nunique()

region_counts = df.groupby("matched_admin1_id")["month_year"].nunique()

print("Expected months per region:", expected_months)
print("\nRegions with missing months:")
print((region_counts < expected_months).sum(), "regions")

print("\nExample incomplete regions:")
print(region_counts[region_counts < expected_months].head())


TIME CONTINUITY CHECK
Expected months per region: 90

Regions with missing months:
0 regions

Example incomplete regions:
Series([], Name: month_year, dtype: int64)
